# ECG Classification — 12SL Features

Continuation of the 53-label experiment. Logistic Regression and Random Forest have already been trained and saved.

Run this notebook from the same project directory and use the same feature CSV and ECG index. The saved LR and RF models are loaded, not retrained. New models and comparison results go into `optimized_results`. The test set (fold 10) is reserved for final testing.

In [1]:
import numpy as np
import pandas as pd
import json
import joblib

from pathlib import Path
from tqdm.auto import tqdm

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score
)

from xgboost import XGBClassifier

import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset


/usr/lib/python3/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset

In [2]:
FEATURE_PATH = Path("data/ptb-xl-plus/features/12sl_features.csv")
INDEX_DIR = Path("data/processed/12sl_median_images_final")

NUM_LABELS = 53
EXPERIMENT_ROOT = Path("data/processed/12sl_feature_models")
DATA_DIR = EXPERIMENT_ROOT / "top_53"
RESULT_DIR = EXPERIMENT_ROOT / "optimized_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

features = pd.read_csv(FEATURE_PATH)
df = pd.read_csv(INDEX_DIR / "training_index_with_folds.csv")

print("Feature data:", features.shape)
print("ECG index:", df.shape)


Feature data: (21799, 783)
ECG index: (20914, 160)


In [3]:
assert features["ecg_id"].notna().all()
assert features["ecg_id"].is_unique

assert df["ecg_id"].is_unique
assert df[["ecg_id", "patient_id", "strat_fold"]].notna().all().all()
assert df["strat_fold"].isin(range(1, 11)).all()

missing_ids = df.loc[~df["ecg_id"].isin(features["ecg_id"]), "ecg_id"]

print("ECGs without features:", len(missing_ids))

assert missing_ids.empty, "Some ECGs are missing from the feature CSV. Check the dataset before comparing models."

features = features.set_index("ecg_id")

assert not any(col.startswith("label_") for col in features.columns)

print("Number of feature columns:", features.shape[1])


ECGs without features: 0
Number of feature columns: 782


## Patient-wise splitting

In [4]:
train_df = df[df["strat_fold"].between(1, 8)].copy()
val_df = df[df["strat_fold"] == 9].copy()
test_df = df[df["strat_fold"] == 10].copy()

assert set(train_df.patient_id).isdisjoint(val_df.patient_id)
assert set(train_df.patient_id).isdisjoint(test_df.patient_id)
assert set(val_df.patient_id).isdisjoint(test_df.patient_id)

assert (len(train_df), len(val_df), len(test_df)) == (16608, 2148, 2158)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))


Training: 16608
Validation: 2148
Testing: 2158


In [5]:
with open(DATA_DIR / "selected_labels.json") as f:
    selected_labels = json.load(f)

label_cols = [
    col for col in df.columns
    if col.startswith("label_")
]

label_counts = train_df[label_cols].sum().sort_values(ascending=False, kind="stable")

labels_5 = label_counts.head(5).index.tolist()
labels_10 = label_counts.head(10).index.tolist()

assert len(selected_labels) == NUM_LABELS
assert set(labels_10).issubset(selected_labels)

print("Selected labels:", selected_labels)

distribution = pd.DataFrame({
    "Train": train_df[selected_labels].sum(),
    "Validation": val_df[selected_labels].sum()
})

assert distribution["Train"].between(1, len(train_df)-1).all()
assert distribution["Validation"].between(1, len(val_df)-1).all()

display(distribution)


Selected labels: ['label_NSR', 'label_AB', 'label_NML', 'label_SBRAD', 'label_SRTH', 'label_ABR', 'label_IMI;AU', 'label_LAD3', 'label_BORDE', 'label_AFIB', 'label_FAV', 'label_LVH2', 'label_SAR', 'label_NST', 'label_PVC', 'label_ST&', 'label_NT', 'label_STACH', 'label_NSTT', 'label_2ST', 'label_LT', 'label_PAC', 'label_RBBB', 'label_SMI;AU', 'label_AMI;AU', 'label_LNGQT', 'label_LBBB', 'label_ASMI;AU', 'label_AFB', 'label_LAE', 'label_IRBBB', 'label_RVR', 'label_LOWV', 'label_IT', 'label_ABER', 'label_MSAR', 'label_QRSW-2ST', 'label_ALT', 'label_RAD', 'label_PSVC', 'label_LMI;AU', 'label_IVCB', 'label_STEMI', 'label_ILT', 'label_AT', 'label_ALMI;AU', 'label_UR', 'label_VPR', 'label_FLUT', 'label_IVCD', 'label_$SABCOND', 'label_BIFB', 'label_QRSW']


,Train,Validation
label_NSR,9641,1253
label_AB,8770,1069
label_NML,4590,635
label_SBRAD,2594,317
label_SRTH,1939,255
label_ABR,1931,256
label_IMI;AU,1714,196
label_LAD3,1495,187
label_BORDE,1317,188
label_AFIB,1071,129


## Load the same preprocessing

In [6]:
with open(DATA_DIR / "feature_columns.json") as f:
    feature_cols = json.load(f)

X_train_df = features.loc[train_df["ecg_id"], feature_cols].copy()
X_val_df = features.loc[val_df["ecg_id"], feature_cols].copy()

X_train_df = X_train_df.apply(pd.to_numeric, errors="raise")
X_val_df = X_val_df.apply(pd.to_numeric, errors="raise")

X_train_df = X_train_df.replace([np.inf, -np.inf], np.nan)
X_val_df = X_val_df.replace([np.inf, -np.inf], np.nan)

print("Training features:", X_train_df.shape)
print("Validation features:", X_val_df.shape)


Training features: (16608, 782)
Validation features: (2148, 782)


In [7]:
imputer = joblib.load(DATA_DIR / "imputer.pkl")
scaler = joblib.load(DATA_DIR / "scaler.pkl")

X_train = imputer.transform(X_train_df).astype(np.float32)
X_val = imputer.transform(X_val_df).astype(np.float32)

X_train_scaled = scaler.transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)

assert np.isfinite(X_train_scaled).all()
assert np.isfinite(X_val_scaled).all()

print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)


Training features: (16608, 782)
Validation features: (2148, 782)


In [8]:
assert train_df[selected_labels].isin([0, 1]).all().all()
assert val_df[selected_labels].isin([0, 1]).all().all()

y_train = train_df[selected_labels].to_numpy(dtype=np.uint8)
y_val = val_df[selected_labels].to_numpy(dtype=np.uint8)

train_ids = train_df["ecg_id"].to_numpy(dtype=np.int64)
val_ids = val_df["ecg_id"].to_numpy(dtype=np.int64)

assert np.array_equal(train_ids, np.load(DATA_DIR / "train_ids.npy"))
assert np.array_equal(val_ids, np.load(DATA_DIR / "validation_ecg_ids.npy"))
assert np.array_equal(y_train, np.load(DATA_DIR / "y_train.npy"))
assert np.array_equal(y_val, np.load(DATA_DIR / "y_val.npy"))

assert np.array_equal(X_train_df.index, train_ids)
assert np.array_equal(X_val_df.index, val_ids)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)


X_train: (16608, 782) y_train: (16608, 53)
X_val: (2148, 782) y_val: (2148, 53)


## Model evaluation

In [9]:
def calculate_metrics(targets, scores, threshold=0.5):
    assert targets.shape == scores.shape

    predictions = (scores >= threshold).astype(int)

    return {
        "Macro AUROC": roc_auc_score(
            targets, scores, average="macro"
        ),

        "Macro AUPRC": average_precision_score(
            targets, scores, average="macro"
        ),

        "Macro F1": f1_score(
            targets, predictions,
            average="macro", zero_division=0
        ),

        "Micro F1": f1_score(
            targets, predictions,
            average="micro", zero_division=0
        )
    }

def print_metrics(metrics):
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")


## Logistic Regression

In [10]:
lr_model = joblib.load(DATA_DIR / "logistic_regression.pkl")

assert len(lr_model.estimators_) == NUM_LABELS
assert lr_model.n_features_in_ == X_train_scaled.shape[1]

print("Saved Logistic Regression model loaded.")


Saved Logistic Regression model loaded.


In [11]:
lr_val_probs = lr_model.predict_proba(X_val_scaled)

lr_metrics = calculate_metrics(y_val, lr_val_probs)

print_metrics(lr_metrics)


Macro AUROC: 0.8832
Macro AUPRC: 0.4582
Macro F1: 0.4444
Micro F1: 0.6969


## Random Forest

In [12]:
rf_model = joblib.load(DATA_DIR / "random_forest.pkl")

assert len(rf_model.estimators_) == NUM_LABELS
assert rf_model.n_features_in_ == X_train.shape[1]

print("Saved Random Forest model loaded.")


Saved Random Forest model loaded.


In [13]:
rf_val_probs = rf_model.predict_proba(X_val)

rf_metrics = calculate_metrics(y_val, rf_val_probs)

print_metrics(rf_metrics)


Macro AUROC: 0.9356
Macro AUPRC: 0.6358
Macro F1: 0.4142
Micro F1: 0.7781


## SVM

The previous SVM did not finish in the supplied notebook. For a faster linear SVM, reduce C and relax the tolerance. Check any convergence warnings and validation performance. The original SVM file, if it finishes later, is not overwritten.

In [14]:
svm_model = OneVsRestClassifier(
    LinearSVC(
        C=0.1,
        dual=False,
        max_iter=2000,
        tol=1e-3,
        random_state=42
    ),
    n_jobs=2,
    verbose=10
)


In [15]:
svm_path = RESULT_DIR / "linear_svm.pkl"

if svm_path.exists():
    svm_model = joblib.load(svm_path)
    print("Saved SVM model loaded.")
else:
    svm_model.fit(X_train_scaled, y_train)

    print("SVM training completed.")

    joblib.dump(svm_model, svm_path)


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:   11.8s
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:   44.3s
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:  2.6min
/home/aarish/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/aarish/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/aarish/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed:  4.3min
/home/aarish/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number o

SVM training completed.


/home/aarish/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
[Parallel(n_jobs=2)]: Done  53 out of  53 | elapsed: 12.4min finished


In [16]:
svm_val_scores = svm_model.decision_function(X_val_scaled)

svm_metrics = calculate_metrics(y_val, svm_val_scores, threshold=0.0)

print_metrics(svm_metrics)


Macro AUROC: 0.8721
Macro AUPRC: 0.4425
Macro F1: 0.4309
Micro F1: 0.6911


## XGBoost

In [17]:
xgb_model = OneVsRestClassifier(
    XGBClassifier(
        n_estimators=50,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        max_bin=64,
        random_state=42,
        n_jobs=2
    ),
    n_jobs=1,
    verbose=10
)


In [18]:
xgb_path = RESULT_DIR / "xgboost.pkl"

if xgb_path.exists():
    xgb_model = joblib.load(xgb_path)
    print("Saved XGBoost model loaded.")
else:
    xgb_model.fit(X_train, y_train)

    print("XGBoost training completed.")

    joblib.dump(xgb_model, xgb_path)


[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    2.5s
[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:    5.4s
[Parallel(n_jobs=1)]: Done   7 tasks      | elapsed:    8.4s
[Parallel(n_jobs=1)]: Done  12 tasks      | elapsed:   13.6s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:   18.9s
[Parallel(n_jobs=1)]: Done  24 tasks      | elapsed:   25.9s
[Parallel(n_jobs=1)]: Done  31 tasks      | elapsed:   32.8s
[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:   42.4s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:   51.7s


XGBoost training completed.


[Parallel(n_jobs=1)]: Done  53 out of  53 | elapsed:   55.9s finished


In [19]:
xgb_val_probs = xgb_model.predict_proba(X_val)

xgb_metrics = calculate_metrics(y_val, xgb_val_probs)

print_metrics(xgb_metrics)


Macro AUROC: 0.9441
Macro AUPRC: 0.6883
Macro F1: 0.5976
Micro F1: 0.8144


## AdaBoost

In [20]:
ada_model = OneVsRestClassifier(
    AdaBoostClassifier(
        n_estimators=50,
        learning_rate=0.5,
        random_state=42
    ),
    n_jobs=2,
    verbose=10
)


In [21]:
ada_path = RESULT_DIR / "adaboost.pkl"

if ada_path.exists():
    ada_model = joblib.load(ada_path)
    print("Saved AdaBoost model loaded.")
else:
    ada_model.fit(X_train, y_train)

    print("AdaBoost training completed.")

    joblib.dump(ada_model, ada_path)


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:   14.1s
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:   28.2s
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:  1.2min
[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed:  1.6min
[Parallel(n_jobs=2)]: Done  21 tasks      | elapsed:  2.6min
[Parallel(n_jobs=2)]: Done  28 tasks      | elapsed:  3.3min
[Parallel(n_jobs=2)]: Done  37 tasks      | elapsed:  4.5min
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:  5.4min
[Parallel(n_jobs=2)]: Done  53 out of  53 | elapsed:  6.3min finished


AdaBoost training completed.


In [22]:
ada_val_probs = ada_model.predict_proba(X_val)

ada_metrics = calculate_metrics(y_val, ada_val_probs)

print_metrics(ada_metrics)


Macro AUROC: 0.9245
Macro AUPRC: 0.5829
Macro F1: 0.4186
Micro F1: 0.7246


## Classical model comparison

In [23]:
model_scores = {
    "Logistic Regression": (lr_val_probs, 0.5),
    "Random Forest": (rf_val_probs, 0.5),
    "SVM": (svm_val_scores, 0.0),
    "XGBoost": (xgb_val_probs, 0.5),
    "AdaBoost": (ada_val_probs, 0.5)
}

results = []

for name, (scores, threshold) in model_scores.items():
    results.append({
        "Model": name,
        **calculate_metrics(y_val, scores, threshold)
    })

classical_results = pd.DataFrame(results)

display(classical_results.round(4))

classical_results.to_csv(RESULT_DIR / "classical_53_results.csv", index=False)


,Model,Macro AUROC,Macro AUPRC,Macro F1,Micro F1
0,Logistic Regression,0.8832,0.4582,0.4444,0.6969
1,Random Forest,0.9356,0.6358,0.4142,0.7781
2,SVM,0.8721,0.4425,0.4309,0.6911
3,XGBoost,0.9441,0.6883,0.5976,0.8144
4,AdaBoost,0.9245,0.5829,0.4186,0.7246


## MLP

Unlike the classical One-vs-Rest models, the MLP learns from all its labels together. Therefore, we train an MLP separately for 53, 10 and 5 labels.

In [24]:
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 0.0003
PATIENCE = 5


Device: cuda


In [25]:
class ECGMLP(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)


In [26]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    running_loss = 0.0

    for features, targets in tqdm(loader, desc="Training", leave=False):
        features = features.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)

        # Forward propagation
        logits = model(features)

        # Calculate loss
        loss = criterion(logits, targets)

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()

        running_loss += loss.item() * features.size(0)

    return running_loss / len(loader.dataset)


In [27]:
def evaluate_model(model, loader, criterion):
    model.eval()

    running_loss = 0.0
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for features, targets in tqdm(
            loader, desc="Evaluating", leave=False
        ):
            features = features.to(device)
            targets = targets.to(device)

            logits = model(features)

            loss = criterion(logits, targets)

            probs = torch.sigmoid(logits)

            running_loss += loss.item() * features.size(0)

            all_targets.append(targets.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    y_true = np.concatenate(all_targets, axis=0)
    y_probs = np.concatenate(all_probs, axis=0)

    y_preds = (y_probs >= 0.5).astype(int)

    metrics = {
        "loss": running_loss / len(loader.dataset),

        "macro_auroc": roc_auc_score(
            y_true, y_probs, average="macro"
        ),

        "macro_auprc": average_precision_score(
            y_true, y_probs, average="macro"
        ),

        "macro_f1": f1_score(
            y_true, y_preds,
            average="macro", zero_division=0
        ),

        "micro_f1": f1_score(
            y_true, y_preds,
            average="micro", zero_division=0
        )
    }

    return metrics, y_true, y_probs


In [28]:
labels_by_count = {
    53: selected_labels,
    10: labels_10,
    5: labels_5
}

mlp_scores = {}


In [29]:
for n, labels in labels_by_count.items():
    print(f"\nMLP — {n} labels")

    torch.manual_seed(42)
    np.random.seed(42)

    indices = [selected_labels.index(label) for label in labels]

    train_targets = y_train[:, indices]
    val_targets = y_val[:, indices]

    train_dataset_mlp = TensorDataset(
        torch.from_numpy(X_train_scaled).float(),
        torch.from_numpy(train_targets).float()
    )

    val_dataset_mlp = TensorDataset(
        torch.from_numpy(X_val_scaled).float(),
        torch.from_numpy(val_targets).float()
    )

    train_loader_mlp = DataLoader(
        train_dataset_mlp,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(device.type == "cuda")
    )

    val_loader_mlp = DataLoader(
        val_dataset_mlp,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    model_mlp = ECGMLP(
        num_features=X_train_scaled.shape[1],
        num_classes=n
    ).to(device)

    criterion_mlp = nn.BCEWithLogitsLoss()

    optimizer_mlp = torch.optim.AdamW(
        model_mlp.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4
    )

    scheduler_mlp = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_mlp, mode="max", factor=0.5, patience=2
    )

    folder = RESULT_DIR / f"top_{n}"
    folder.mkdir(exist_ok=True)

    with open(folder / "selected_labels.json", "w") as f:
        json.dump(labels, f, indent=4)

    BEST_MODEL_PATH_MLP = folder / "mlp_best.pt"

    if not BEST_MODEL_PATH_MLP.exists() or not (folder / "mlp_training_history.csv").exists():
        history_mlp = []
        best_auprc_mlp = -np.inf
        epochs_without_improvement_mlp = 0

        for epoch in range(EPOCHS):
            train_loss_mlp = train_one_epoch(
                model_mlp, train_loader_mlp,
                optimizer_mlp, criterion_mlp
            )

            val_metrics_mlp, _, _ = evaluate_model(model_mlp, val_loader_mlp, criterion_mlp)

            history_mlp.append({
                "epoch": epoch + 1,
                "train_loss": train_loss_mlp,
                **val_metrics_mlp
            })

            scheduler_mlp.step(val_metrics_mlp["macro_auprc"])

            print(
                f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
                f"Train Loss: {train_loss_mlp:.4f} | "
                f"Val Loss: {val_metrics_mlp['loss']:.4f} | "
                f"AUPRC: {val_metrics_mlp['macro_auprc']:.4f} | "
                f"Macro F1: {val_metrics_mlp['macro_f1']:.4f}"
            )

            if val_metrics_mlp["macro_auprc"] > best_auprc_mlp:
                best_auprc_mlp = val_metrics_mlp["macro_auprc"]
                epochs_without_improvement_mlp = 0

                torch.save(model_mlp.state_dict(), BEST_MODEL_PATH_MLP)

                print("Best model saved.")
            else:
                epochs_without_improvement_mlp += 1

            if epochs_without_improvement_mlp >= PATIENCE:
                print("Early stopping triggered.")
                break

        pd.DataFrame(history_mlp).to_csv(folder / "mlp_training_history.csv", index=False)

    model_mlp.load_state_dict(torch.load(BEST_MODEL_PATH_MLP, map_location=device))

    _, targets, probs = evaluate_model(model_mlp, val_loader_mlp, criterion_mlp)

    assert np.array_equal(targets, val_targets)

    mlp_scores[n] = probs

    print_metrics(calculate_metrics(val_targets, probs))



MLP — 53 labels


Epoch [01/30] | Train Loss: 0.2123 | Val Loss: 0.1240 | AUPRC: 0.2618 | Macro F1: 0.0963
Best model saved.


Epoch [02/30] | Train Loss: 0.1277 | Val Loss: 0.1096 | AUPRC: 0.3447 | Macro F1: 0.1549
Best model saved.


Epoch [03/30] | Train Loss: 0.1154 | Val Loss: 0.1019 | AUPRC: 0.3852 | Macro F1: 0.1962
Best model saved.


Epoch [04/30] | Train Loss: 0.1083 | Val Loss: 0.0974 | AUPRC: 0.4076 | Macro F1: 0.2274
Best model saved.


Epoch [05/30] | Train Loss: 0.1032 | Val Loss: 0.0940 | AUPRC: 0.4297 | Macro F1: 0.2545
Best model saved.


... (84 outputs left out)

Epoch [28/30] | Train Loss: 0.1563 | Val Loss: 0.2484 | AUPRC: 0.8328 | Macro F1: 0.7850


Epoch [29/30] | Train Loss: 0.1552 | Val Loss: 0.2485 | AUPRC: 0.8323 | Macro F1: 0.7851


Epoch [30/30] | Train Loss: 0.1521 | Val Loss: 0.2489 | AUPRC: 0.8309 | Macro F1: 0.7845


Macro AUROC: 0.9251
Macro AUPRC: 0.8341
Macro F1: 0.7854
Micro F1: 0.8650


## Final comparison — 5, 10 and 53 labels

We use the corresponding output columns of the trained One-vs-Rest models. No extra fitting is required. MLP predictions come from the separate model trained for each label count.

In [30]:
final_results = []
common_results = []
per_label_results = []

for n, labels in labels_by_count.items():
    indices = [selected_labels.index(label) for label in labels]
    common_indices = [labels.index(label) for label in labels_5]

    targets = y_val[:, indices]

    scores_n = {
        name: (scores[:, indices], threshold)
        for name, (scores, threshold) in model_scores.items()
    }

    scores_n["MLP"] = (mlp_scores[n], 0.5)

    folder = RESULT_DIR / f"top_{n}"

    for name, (scores, threshold) in scores_n.items():
        metrics = calculate_metrics(targets, scores, threshold)

        final_results.append({
            "Number of labels": n,
            "Model": name,
            **metrics
        })

        common_results.append({
            "Number of labels": n,
            "Model": name,
            **calculate_metrics(targets[:, common_indices], scores[:, common_indices], threshold)
        })

        preds = (scores >= threshold).astype(int)

        for i, label in enumerate(labels):
            per_label_results.append({
                "Number of labels": n,
                "Model": name,
                "Label": label,
                "Validation positives": int(targets[:, i].sum()),
                "AUROC": roc_auc_score(targets[:, i], scores[:, i]),
                "AUPRC": average_precision_score(targets[:, i], scores[:, i]),
                "F1": f1_score(targets[:, i], preds[:, i], zero_division=0)
            })

        file_name = name.lower().replace(" ", "_")

        np.savez(
            folder / f"{file_name}_validation.npz",
            scores=scores,
            targets=targets,
            labels=np.asarray(labels),
            ecg_ids=val_ids,
            threshold=threshold
        )

final_val_results = pd.DataFrame(final_results)
common_comparison = pd.DataFrame(common_results)
per_label_results = pd.DataFrame(per_label_results)

display(final_val_results.round(4))

final_val_results.to_csv(RESULT_DIR / "all_label_experiments.csv", index=False)
common_comparison.to_csv(RESULT_DIR / "common_top5_comparison.csv", index=False)
per_label_results.to_csv(RESULT_DIR / "per_label_validation_results.csv", index=False)


,Number of labels,Model,Macro AUROC,Macro AUPRC,Macro F1,Micro F1
0,53,Logistic Regression,0.8832,0.4582,0.4444,0.6969
1,53,Random Forest,0.9356,0.6358,0.4142,0.7781
2,53,SVM,0.8721,0.4425,0.4309,0.6911
3,53,XGBoost,0.9441,0.6883,0.5976,0.8144
4,53,AdaBoost,0.9245,0.5829,0.4186,0.7246
5,53,MLP,0.9269,0.5357,0.4080,0.7361
6,10,Logistic Regression,0.9213,0.7382,0.6905,0.8009
7,10,Random Forest,0.9550,0.8710,0.7866,0.8720
8,10,SVM,0.9201,0.7370,0.6651,0.7948
9,10,XGBoost,0.9577,0.8700,0.8057,0.8772


In [31]:
display(common_comparison.round(4))

for n in [5, 10, 53]:
    print(f"\n{n} Labels")

    display(final_val_results[
        final_val_results["Number of labels"] == n
    ].sort_values("Macro AUPRC", ascending=False).round(4))


,Number of labels,Model,Macro AUROC,Macro AUPRC,Macro F1,Micro F1
0,53,Logistic Regression,0.9025,0.7902,0.7548,0.8428
1,53,Random Forest,0.9371,0.8634,0.8087,0.8954
2,53,SVM,0.8999,0.7899,0.7333,0.8393
3,53,XGBoost,0.9423,0.8682,0.8274,0.8964
4,53,AdaBoost,0.8927,0.7815,0.7759,0.8344
5,53,MLP,0.9234,0.8276,0.7759,0.8634
6,10,Logistic Regression,0.9025,0.7902,0.7548,0.8428
7,10,Random Forest,0.9371,0.8634,0.8087,0.8954
8,10,SVM,0.8999,0.7899,0.7333,0.8393
9,10,XGBoost,0.9423,0.8682,0.8274,0.8964



5 Labels


,Number of labels,Model,Macro AUROC,Macro AUPRC,Macro F1,Micro F1
15,5,XGBoost,0.9423,0.8682,0.8274,0.8964
13,5,Random Forest,0.9371,0.8634,0.8087,0.8954
17,5,MLP,0.9251,0.8341,0.7854,0.8650
12,5,Logistic Regression,0.9025,0.7902,0.7548,0.8428
14,5,SVM,0.8999,0.7899,0.7333,0.8393
16,5,AdaBoost,0.8927,0.7815,0.7759,0.8344



10 Labels


,Number of labels,Model,Macro AUROC,Macro AUPRC,Macro F1,Micro F1
7,10,Random Forest,0.9550,0.8710,0.7866,0.8720
9,10,XGBoost,0.9577,0.8700,0.8057,0.8772
11,10,MLP,0.9458,0.8210,0.7664,0.8418
10,10,AdaBoost,0.9210,0.7721,0.6614,0.8002
6,10,Logistic Regression,0.9213,0.7382,0.6905,0.8009
8,10,SVM,0.9201,0.7370,0.6651,0.7948



53 Labels


,Number of labels,Model,Macro AUROC,Macro AUPRC,Macro F1,Micro F1
3,53,XGBoost,0.9441,0.6883,0.5976,0.8144
1,53,Random Forest,0.9356,0.6358,0.4142,0.7781
4,53,AdaBoost,0.9245,0.5829,0.4186,0.7246
5,53,MLP,0.9269,0.5357,0.4080,0.7361
0,53,Logistic Regression,0.8832,0.4582,0.4444,0.6969
2,53,SVM,0.8721,0.4425,0.4309,0.6911


## Saved models

The existing models and preprocessing stay in `top_53`. New SVM, XGBoost and AdaBoost models are stored under `optimized_results`, along with the separate MLP checkpoints. Rerunning uses saved new models if they exist; use a new `RESULT_DIR` if you change the training data or model settings. Keep the original feature CSV unchanged. The results are validation scores for reproducing automatic 12SL statements, not independent clinical diagnoses.

In [32]:
model_paths = [
    DATA_DIR / "logistic_regression.pkl",
    DATA_DIR / "random_forest.pkl",
    RESULT_DIR / "linear_svm.pkl",
    RESULT_DIR / "xgboost.pkl",
    RESULT_DIR / "adaboost.pkl"
]

for n in [5, 10, 53]:
    model_paths.append(RESULT_DIR / f"top_{n}" / "mlp_best.pt")

for path in model_paths:
    print(path, ":", path.exists())


data/processed/12sl_feature_models/top_53/logistic_regression.pkl : True
data/processed/12sl_feature_models/top_53/random_forest.pkl : True
data/processed/12sl_feature_models/optimized_results/linear_svm.pkl : True
data/processed/12sl_feature_models/optimized_results/xgboost.pkl : True
data/processed/12sl_feature_models/optimized_results/adaboost.pkl : True
data/processed/12sl_feature_models/optimized_results/top_5/mlp_best.pt : True
data/processed/12sl_feature_models/optimized_results/top_10/mlp_best.pt : True
data/processed/12sl_feature_models/optimized_results/top_53/mlp_best.pt : True
